In [231]:
from pyspark. sql import SparkSession
from pyspark. sql. functions import col, udf
from pyspark. sql.types import StringType, IntegerType
import time



...

Ellipsis

In [232]:
spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()


In [82]:
df = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv("/workspace/data.csv")

df.show(10)

+---+-------+----------+------+---+
| ID|   Name|Department|Salary|Age|
+---+-------+----------+------+---+
|  1|  Aarav|        IT| 65000| 24|
|  2| Vivaan|        HR| 52000| 29|
|  3| Aditya|   Finance| 70000| 31|
|  4|Krishna|        IT| 80000| 35|
|  5| Ishaan| Marketing| 55000| 27|
|  6|  Arjun|     Sales| 60000| 30|
|  7|  Rohan|   Finance| 72000| 33|
|  8|  Kabir|        IT| 68000| 26|
|  9|Reyansh|        HR| 50000| 28|
| 10| Atharv| Marketing| 58000| 32|
+---+-------+----------+------+---+
only showing top 10 rows



In [83]:
df.take(10)



[Row(ID=1, Name='Aarav', Department='IT', Salary=65000, Age=24),
 Row(ID=2, Name='Vivaan', Department='HR', Salary=52000, Age=29),
 Row(ID=3, Name='Aditya', Department='Finance', Salary=70000, Age=31),
 Row(ID=4, Name='Krishna', Department='IT', Salary=80000, Age=35),
 Row(ID=5, Name='Ishaan', Department='Marketing', Salary=55000, Age=27),
 Row(ID=6, Name='Arjun', Department='Sales', Salary=60000, Age=30),
 Row(ID=7, Name='Rohan', Department='Finance', Salary=72000, Age=33),
 Row(ID=8, Name='Kabir', Department='IT', Salary=68000, Age=26),
 Row(ID=9, Name='Reyansh', Department='HR', Salary=50000, Age=28),
 Row(ID=10, Name='Atharv', Department='Marketing', Salary=58000, Age=32)]

In [69]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 1


In [84]:
df_repartitioned =df.repartition(20)


In [85]:
print("After Repartitions:",df.repartition(25))

After Repartitions: DataFrame[ID: int, Name: string, Department: string, Salary: int, Age: int]


In [ ]:
partition_sizes = df_repartitioned.rdd.glom().map(len).collect()

print(partition_sizes)

In [86]:
df_coalesced = df_repartitioned.coalesce(20)

In [87]:
print(df_repartitioned.rdd.getNumPartitions())

20


In [90]:
df_repartitioned.write \
  .mode("overwrite") \
  .option("header", True) \
  .csv("employee_csv")

In [92]:
spark.stop()


In [93]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType, IntegerType
import time


In [94]:

spark = SparkSession.builder.appName("AdvanceSparkApp").getOrCreate()

df = spark.range(0, 100000).withColumn("value", col("id") % 1000)


In [95]:
df.take(10)

[Row(id=0, value=0),
 Row(id=1, value=1),
 Row(id=2, value=2),
 Row(id=3, value=3),
 Row(id=4, value=4),
 Row(id=5, value=5),
 Row(id=6, value=6),
 Row(id=7, value=7),
 Row(id=8, value=8),
 Row(id=9, value=9)]

In [96]:
print("Before Partitions:", df.rdd.getNumPartitions())

Before Partitions: 2


In [ ]:
df_repartitioned = df.repartition(11)

In [98]:
df_coalesced = df_repartitioned.coalesce(2)

In [99]:
print("After Coalesced:", df_coalesced.rdd.getNumPartitions())


After Coalesced: 2


In [141]:
optimized_df = df.filter(col("value") > 500).filter(col("id") < 5000000).select("id", "value")

In [142]:
optimized_df.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- InMemoryTableScan [id#612L, value#614L]
      +- InMemoryRelation [id#612L, value#614L], StorageLevel(disk, memory, deserialized, 1 replicas)
            +- *(1) Project [id#612L, (id#612L % 1000) AS value#614L]
               +- *(1) Filter (((id#612L % 1000) > 500) AND (id#612L < 5000000))
                  +- *(1) Range (0, 100000, step=1, splits=2)




In [143]:
start_time = time.time()
count_uncached = optimized_df.count() # Action triggers DAG
end_time = time. time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 49900 | Time: 0.0783 seconds


In [149]:
optimized_df.cache()

26/06/13 06:11:15 WARN CacheManager: Asked to cache already cached data.


DataFrame[id: bigint, value: bigint]

In [223]:
start_time = time.time()
count_cached = optimized_df.count() # Action triggers DAG
end_time = time. time()
print(f"1. Optimized Execution | Count: {count_uncached} | Time: {round(end_time - start_time, 4)} seconds")

1. Optimized Execution | Count: 49900 | Time: 0.0375 seconds


In [195]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]

In [226]:
spark.stop()

In [243]:
data = [("Alice", 25), ("Bob", 17), ("Charlie", 35), ("David", 15)]
df1 = spark.createDataFrame(data, ["Name", "Age"])

In [244]:
df1.show()

+-------+---+
|   Name|Age|
+-------+---+
|  Alice| 25|
|    Bob| 17|
|Charlie| 35|
|  David| 15|
+-------+---+



In [245]:
def categorize_age(age):
    if age >= 18:
        return "Adult"
    return "Minor"

In [246]:
age_category_udf = udf(categorize_age, StringType())

In [248]:
df_method1 = df1.withColumn("Category", age_category_udf(col("Age")))
print("Method 1: Standard UDF via DataFrame API")
df_method1.show()

Method 1: Standard UDF via DataFrame API


+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [249]:
spark.udf.register("sql_categorize_age", categorize_age, StringType())

df1.createOrReplaceTempView("people")

In [250]:
sql_df = spark.sql("""
    SELECT
       Name,
       Age, 
       sql_categorize_age(Age) AS Category
   FROM people
""")

sql_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 25|   Adult|
|    Bob| 17|   Minor|
|Charlie| 35|   Adult|
|  David| 15|   Minor|
+-------+---+--------+



In [251]:
def can_drive(age):
    if age >= 18:
        return "Yes"
    return "No"

In [252]:
can_drive_udf = udf(can_drive, StringType())

In [253]:
df_drive = df1.withColumn(
    "Can_Drive",
    can_drive_udf(col("Age"))
)

df_drive.show()

+-------+---+---------+
|   Name|Age|Can_Drive|
+-------+---+---------+
|  Alice| 25|      Yes|
|    Bob| 17|       No|
|Charlie| 35|      Yes|
|  David| 15|       No|
+-------+---+---------+



In [254]:
spark.udf.register(
    "sql_can_drive",
    can_drive,
    StringType()
)

df1.createOrReplaceTempView("people")

In [255]:

sql_df = spark.sql("""
SELECT
    Name,
    Age,
    sql_can_drive(Age) AS Can_Drive
FROM people
""")

sql_df.show()

+-------+---+---------+
|   Name|Age|Can_Drive|
+-------+---+---------+
|  Alice| 25|      Yes|
|    Bob| 17|       No|
|Charlie| 35|      Yes|
|  David| 15|       No|
+-------+---+---------+

